<a href="https://colab.research.google.com/github/MallikaReddy-hub/prj495-mental-chatbot/blob/main/training/01_prepare_data_and_translate.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 - Dataset preparation and translation (PRJ_495)

Builds the training data for the multilingual mental health support chatbot.

**Pipeline**
1. Load `nbertagnolli/counsel-chat` (English counselor answers to forum questions)
2. Clean: remove diagnosing/prescribing answers, forum chatter, links, very short/long rows; keep best-voted answer per question
3. Translate a 100-example sample into Hindi, Telugu, Tamil, Kannada and Spanish with NLLB-200
4. Add hand-written safety examples (decline diagnosis / medication, redirect to a professional) in all six languages
5. Merge everything into `mh_train.jsonl` and `mh_val.jsonl` (chat format with a fixed system prompt)

Run on a Colab T4 GPU. Outputs are cleared on purpose: the source posts are real people's forum text, and the generated files are saved to Google Drive, not committed to this repo.

In [1]:
!pip install -q -U transformers datasets accelerate sentencepiece
from google.colab import drive
drive.mount('/content/drive')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 12.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.
Mounted at /content/drive


## 1. Load and clean counsel-chat
Add your Hugging Face token as a Colab secret named `HF_TOKEN` (never paste it into a cell).

In [2]:
import re, json, random
from datasets import load_dataset
from huggingface_hub import login
from google.colab import userdata

login(userdata.get("HF_TOKEN"))
raw = load_dataset("nbertagnolli/counsel-chat")["train"]
print(raw.column_names, len(raw))

README.md:   0%|          | 0.00/4.92k [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


20220401_counsel_chat.csv:   0%|          | 0.00/4.13M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2775 [00:00<?, ? examples/s]

['questionID', 'questionTitle', 'questionText', 'questionLink', 'topic', 'therapistInfo', 'therapistURL', 'answerText', 'upvotes', 'views'] 2775


In [3]:
# Answers that diagnose or prescribe
BAD = re.compile(r"\byou (have|are suffering from) (depression|bipolar|anxiety disorder|ptsd|adhd|ocd)\b"
                 r"|\b(take|try|start) (\d+ ?mg|medication|antidepressants?)\b", re.I)
# Forum chatter, credentials, links, phone numbers
FORUM = re.compile(r"other therapists|the comments|as a (certified|licensed)|i am a (certified|licensed)"
                   r"|my (website|practice|office)|https?://|www\.|@\w+\.\w+|\b\d{3}[-. ]\d{3}[-. ]\d{4}\b", re.I)

ds = raw.map(lambda ex: {
    "Context": f"{ex['questionTitle'] or ''}. {ex['questionText'] or ''}".strip(),
    "Response": (ex["answerText"] or "").strip()})

ds = ds.filter(lambda e: 40 < len(e["Context"]) < 1500
                         and 80 < len(e["Response"]) < 2000
                         and not BAD.search(e["Response"])
                         and not FORUM.search(e["Response"]))

# keep the best-voted answer per question
seen, keep = set(), []
for i in sorted(range(len(ds)), key=lambda i: -(ds[i]["upvotes"] or 0)):
    if ds[i]["questionText"] not in seen:
        seen.add(ds[i]["questionText"]); keep.append(i)
ds = ds.select(keep)
print("usable English examples:", len(ds))

split = ds.train_test_split(test_size=0.08, seed=42)
train_ds, val_ds = split["train"], split["test"]

Map:   0%|          | 0/2775 [00:00<?, ? examples/s]

Filter:   0%|          | 0/2775 [00:00<?, ? examples/s]

usable English examples: 789


## 2. Translate a sample with NLLB-200
Takes roughly an hour on a T4 for 100 examples x 5 languages. Each language is saved to Drive as it finishes. Re-running overwrites those files, so only run it if you need to rebuild the data.

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, logging
logging.set_verbosity_error()

NLLB = "facebook/nllb-200-distilled-600M"
ntok = AutoTokenizer.from_pretrained(NLLB, src_lang="eng_Latn")
nmodel = AutoModelForSeq2SeqLM.from_pretrained(NLLB, torch_dtype=torch.float16).to("cuda")

def split_sents(t):
    return [s.strip() for s in re.split(r'(?<=[.!?])\s+', t) if s.strip()]

def translate_doc(text, lang, bs=24):
    sents, out = split_sents(text), []
    for i in range(0, len(sents), bs):
        enc = ntok(sents[i:i+bs], return_tensors="pt", padding=True,
                   truncation=True, max_length=200).to("cuda")
        with torch.no_grad():
            gen = nmodel.generate(**enc, max_new_tokens=256, max_length=None, num_beams=2,
                                  forced_bos_token_id=ntok.convert_tokens_to_ids(lang))
        out += ntok.batch_decode(gen, skip_special_tokens=True)
    return " ".join(out)

LANGS = {"hi": "hin_Deva", "te": "tel_Telu", "ta": "tam_Taml", "kn": "kan_Knda", "es": "spa_Latn"}
N = 100
sample = train_ds.shuffle(seed=1).select(range(N))

for code, nllb_code in LANGS.items():
    with open(f"/content/drive/MyDrive/mh_{code}.jsonl", "w", encoding="utf-8") as f:
        for ex in sample:
            f.write(json.dumps({"lang": code,
                                "Context": translate_doc(ex["Context"], nllb_code),
                                "Response": translate_doc(ex["Response"], nllb_code)},
                               ensure_ascii=False) + "\n")
    print(code, "done")

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 4.85MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.3MB            

...

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.46GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

hi done
te done
ta done
kn done
es done


## 3. Hand-written safety examples
The bot must decline to diagnose or recommend medicine and redirect to a professional. Crisis / self-harm messages are deliberately NOT here: they are handled by a fixed rule-based safety layer before the model.

In [5]:
SAFETY_EN = [
 {"Context": "I've been feeling low for weeks and can't enjoy anything. Do I have depression?",
  "Response": "I'm sorry you've been carrying this for weeks, that sounds exhausting. I can't diagnose conditions, and only a doctor or mental health professional can say what is going on. What I can do is listen and help you think through what has been happening. A professional can also look at the full picture, so it may be worth booking a visit with a doctor or counselor. In the meantime, could you tell me when you first noticed things changing?"},
 {"Context": "Which medicine should I take for my anxiety? Just tell me a name.",
  "Response": "I understand wanting something that works quickly when anxiety is hard. I'm not able to recommend medicines or doses, because the right choice depends on your health and needs a doctor or psychiatrist to decide. What I can do is share a few ways to calm yourself in the moment, such as slow breathing or grounding with things you can see and hear around you. Would you like to try one together?"},
 {"Context": "My mood goes very high and then very low. I think I'm bipolar, right?",
  "Response": "It sounds like these swings are hard to live with, and I'm glad you're paying attention to them. I can't tell you whether it's bipolar disorder or anything else, since that takes a proper assessment by a professional. It could help to keep a simple note of your mood each day and bring it to a doctor or psychologist. Would you like to talk about what the high and low periods feel like?"},
 {"Context": "Can you be my therapist? I don't want to go to a real one.",
  "Response": "Thank you for trusting me with this. I'm happy to be a supportive space to talk and to share coping ideas, but I'm not a therapist and can't replace one. A trained person can follow your situation over time in a way I can't. If something about going feels scary or expensive, we can talk through that too. What's making it feel hard to go?"},
 {"Context": "I can't sleep. How many sleeping pills can I take to finally rest?",
  "Response": "Not sleeping is really draining, and I'm sorry you're going through it. I can't suggest how many pills to take, and taking medicine without a doctor's guidance can be unsafe, so please check with a doctor or pharmacist about what is right for you. Meanwhile, some things that help many people are a fixed wake-up time, keeping screens away for an hour before bed, and a slow breathing routine. Would you like me to walk you through one tonight?"},
 {"Context": "My friend is always sad and angry. Can you tell me what illness he has?",
  "Response": "It's kind of you to worry about your friend. I can't tell what is going on with him, and nobody can from the outside. What often helps is letting him know you care, listening without trying to fix everything, and gently suggesting he talk to a counselor or doctor. Would you like some ideas on how to start that conversation?"},
 {"Context": "I keep losing focus and forgetting things. Do I have ADHD?",
  "Response": "That sounds frustrating, especially when it affects your work or studies. I can't say whether it's ADHD, because stress, poor sleep and many other things can cause the same signs, and only a professional can assess it. Writing down when it happens most can help you describe it clearly to a doctor. Would you like to go through some focus strategies in the meantime?"},
 {"Context": "I feel better now so I want to stop taking my medication. Is that okay?",
  "Response": "I'm glad you're feeling better. Please don't stop your medication on your own, because stopping suddenly can sometimes cause problems, and that decision belongs with the doctor who prescribed it. You could tell them just what you told me, that you feel better and want to stop, and they can guide you safely. Would it help to think about what to ask at that visit?"},
 {"Context": "I have exams next week and I feel like I'm going to break down. I can't think straight.",
  "Response": "That pressure is heavy, and feeling overwhelmed before exams is very common. Try pausing for a minute: breathe in for four counts, out for six, a few times. Then pick just one small topic to study for 25 minutes, with a short break after. Breaking it into small pieces makes it feel less huge. What subject is worrying you most?"},
 {"Context": "Everyone says I overthink everything. Is something wrong with me?",
  "Response": "Feeling like your mind won't stop can be tiring, and it doesn't mean something is wrong with you. I can't diagnose anything, but I can help you work with the habit. One technique is to write the worry down, then ask whether it's something you can act on today. If it is, note one small step, and if not, let it sit for now. What does your overthinking usually focus on?"},
]

with open("/content/drive/MyDrive/mh_safety_en.jsonl", "w", encoding="utf-8") as f:
    for r in SAFETY_EN:
        f.write(json.dumps({"lang": "en", **r}, ensure_ascii=False) + "\n")

for code, nllb_code in LANGS.items():
    with open(f"/content/drive/MyDrive/mh_safety_{code}.jsonl", "w", encoding="utf-8") as f:
        for ex in SAFETY_EN:
            f.write(json.dumps({"lang": code,
                                "Context": translate_doc(ex["Context"], nllb_code),
                                "Response": translate_doc(ex["Response"], nllb_code)},
                               ensure_ascii=False) + "\n")
    print(code, "safety done")

hi safety done
te safety done
ta safety done
kn safety done
es safety done


## 4. Merge into the final training and validation files
Chat format with a fixed system prompt. 5 translated examples per language and 2 safety examples per language are held out for validation; the remaining safety examples are repeated 3x in training so the behaviour sticks.

In [6]:
SYSTEM = ("You are a supportive, multilingual mental health assistant. "
          "Reply in the same language as the user. Do not diagnose or prescribe.")

def row(lang, c, r):
    return {"lang": lang, "messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": c},
        {"role": "assistant", "content": r}]}

train, val = [], []
for ex in train_ds: train.append(row("en", ex["Context"], ex["Response"]))
for ex in val_ds:   val.append(row("en", ex["Context"], ex["Response"]))

for code in ["hi", "te", "ta", "kn", "es"]:
    rows = [json.loads(l) for l in open(f"/content/drive/MyDrive/mh_{code}.jsonl", encoding="utf-8")]
    random.Random(1).shuffle(rows)
    for r in rows[:5]:  val.append(row(code, r["Context"], r["Response"]))
    for r in rows[5:]:  train.append(row(code, r["Context"], r["Response"]))

for code in ["en", "hi", "te", "ta", "kn", "es"]:
    rows = SAFETY_EN if code == "en" else [json.loads(l) for l in open(f"/content/drive/MyDrive/mh_safety_{code}.jsonl", encoding="utf-8")]
    for r in rows[:2]:  val.append(row(code, r["Context"], r["Response"]))
    for _ in range(3):
        for r in rows[2:]: train.append(row(code, r["Context"], r["Response"]))

random.Random(7).shuffle(train); random.Random(7).shuffle(val)

for name, data in [("train", train), ("val", val)]:
    with open(f"/content/drive/MyDrive/mh_{name}.jsonl", "w", encoding="utf-8") as f:
        for r in data: f.write(json.dumps(r, ensure_ascii=False) + "\n")

from collections import Counter
print("train:", len(train), dict(Counter(r["lang"] for r in train)))
print("val:  ", len(val),   dict(Counter(r["lang"] for r in val)))

train: 1344 {'hi': 119, 'kn': 119, 'en': 749, 'ta': 119, 'te': 119, 'es': 119}
val:   101 {'en': 66, 'es': 7, 'kn': 7, 'te': 7, 'ta': 7, 'hi': 7}
